In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
import torch
import zipfile
import os

In [2]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [4]:
zip_path = "/home/ankitanand/Documents/pp/Finetuning_HF/Preference_Alignment_data/tinyllama-preference-alignment.zip"

# Extract all zip files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [5]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-instruction/checkpoint-12"

In [6]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [7]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [8]:
inputs = tokenizer(prompt ,return_tensors="pt").to("cuda")  

In [9]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [10]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
How can machine learning help us to improve the efficiency of our work? How do we use it in everyday life? Why does this technology interest so many people nowadays? Discuss!
You have a very diverse background, as you worked for many years at Google. Could you tell us more about your experience with Artificial Intelligence?
I'm passionate about AI and its potential to create new business models and disrupt the existing ones. In my research I have also been looking at the social impact of this technology: what are the ethical challenges? Can we build a just society based on AI or not? What kind of regulations should be put into place to protect human rights and the environment from harmful impacts of AI? How do we apply ethics when developing AI solutions? What are the barriers to their implementation?
AI is really exciting because it allows us to do things that

In [11]:
instruction_checkpoint = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-instruction/checkpoint-12"

In [17]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [12]:
# Load_dataset
dataset = load_dataset("csv", data_files="/home/ankitanand/Documents/pp/Finetuning_HF/Preference_Alignment_data/pharma_preference_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [18]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [19]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [20]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

### Step A: Load base

In [21]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Step B: Load Instruction LoRA + merge

In [22]:
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [24]:
model = model.merge_and_unload()

### Step C: Attach NEW LoRA for preference

In [25]:
pref_model_lora = get_peft_model(model, lora_config)

In [26]:
os.environ["WANDB_DISABLED"] = "true" # this is because we dont want to capture any metrics

In [32]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-prefernece-alignement",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    beta=0.1,
    report_to="none",
    loss_type="sigmoid",
    remove_unused_columns=False
)

In [34]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

In [35]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


TrainOutput(global_step=4, training_loss=0.6920966506004333, metrics={'train_runtime': 84.0481, 'train_samples_per_second': 0.238, 'train_steps_per_second': 0.048, 'total_flos': 17467419820032.0, 'train_loss': 0.6920966506004333, 'entropy': 2.0759711027145387, 'num_tokens': 2264.0, 'logits/chosen': -3.573224237233929, 'logits/rejected': -3.6676443818686666, 'mean_token_accuracy': 0.5934551686048508, 'rewards/chosen': 0.0048234557500109075, 'rewards/rejected': 0.002660102898880723, 'rewards/accuracies': 0.45, 'rewards/margins': 0.0021633528638631107, 'logps/chosen': -93.57769622802735, 'logps/rejected': -63.100325393676755, 'epoch': 4.0})

## Testing with Non-Instruction Model

In [36]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

## Testing with Non-Instruction Model

In [37]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-instruction/checkpoint-12"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [38]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [39]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [40]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Crohn's disease (CD) is an inflammatory bowel disease that can be severe. Recent research has shown that a compound called Metformin can help lower cholesterol, blood sugar, and improve Crohn's symptoms.
Why does Metformin reduce cholesterol?
Metformin is a diuretic that is commonly prescribed to treat high blood sugar levels.
Researchers are interested in metformin because it helps lower blood sugar.
They also think metformin could be useful for reducing cholesterol.
Metformin lowers blood sugar by increasing the amount of insulin released by the pancreas. This may help reduce fat deposits on the liver, which can lead to inflammation and heart disease.
According to the National Institute of Diabetes and Digestive and Kidney Diseases, metformin can decrease triglycerides in people with type 2 diabetes. It also appears to have other benefits, s

## Testing with DPO (Preference-Aligned) Model

In [41]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-prefernece-alignement/checkpoint-4"

In [42]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [43]:
preference_aligned_model.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [44]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [46]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [47]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Karen Rice is a freelance science journalist who has covered health news for more than 10 years, specializing in infectious diseases. She has worked as an editorial assistant at Medscape, the Journal of Clinical Gastroenterology, and Drug Topics, among others. Rice has written for numerous outlets, including The Washington Post, The New York Times, NBC News, and Discover Health, where she served as editor-at-large from 2015 to 2016. In addition, she holds a Bachelor of Science degree in psychobiology from Emory University.
Disclaimer: Karen Rice does not work for, consult, own shares in or receive funding from any company or organization that would benefit from this article, and has disclosed no relevant affiliations beyond their academic appointment.
